In [2]:

import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("dirty_cafe_sales.csv")

# Display first 5 rows
df.head()


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [3]:

print("Dataset Shape:", df.shape)

print("\nDataset Information:")
df.info()

print("\nFirst 5 Rows:")
display(df.head())


Dataset Shape: (10000, 8)

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB

First 5 Rows:


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [4]:

for column in df.columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False).head(15))



--- Transaction ID ---
Transaction ID
TXN_1961373    1
TXN_4977031    1
TXN_4271903    1
TXN_7034554    1
TXN_3160411    1
TXN_2602893    1
TXN_4433211    1
TXN_6699534    1
TXN_4717867    1
TXN_2064365    1
TXN_2548360    1
TXN_3051279    1
TXN_7619095    1
TXN_9437049    1
TXN_8915701    1
Name: count, dtype: int64

--- Item ---
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64

--- Quantity ---
Quantity
5          2013
2          1974
4          1863
3          1849
1          1822
UNKNOWN     171
ERROR       170
NaN         138
Name: count, dtype: int64

--- Price Per Unit ---
Price Per Unit
3.0        2429
4.0        2331
2.0        1227
5.0        1204
1.0        1143
1.5        1133
ERROR       190
NaN         179
UNKNOWN     164
Name: count, dtype: int64

--- Total Spent ---
Total Spent
6.0     979
12.0    939
3

In [5]:

# Create a copy of original dataset
clean_df = df.copy()

# Replace invalid values with NaN
clean_df.replace(['ERROR', 'UNKNOWN'], np.nan, inplace=True)

print("Invalid values replaced successfully!")


Invalid values replaced successfully!


In [6]:

# Check missing values in each column
missing_values = clean_df.isnull().sum()

print("Missing Values:")
print(missing_values)

print("\nTotal Missing Values:", clean_df.isnull().sum().sum())


Missing Values:
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

Total Missing Values: 10082


In [7]:

missing_percentage = (
    clean_df.isnull().sum() / len(clean_df)
) * 100

missing_report = pd.DataFrame({
    'Missing Values': clean_df.isnull().sum(),
    'Missing Percentage': missing_percentage.round(2)
})

display(missing_report)


,Missing Values,Missing Percentage
Transaction ID,0,0.00
Item,969,9.69
Quantity,479,4.79
Price Per Unit,533,5.33
Total Spent,502,5.02
Payment Method,3178,31.78
Location,3961,39.61
Transaction Date,460,4.60


In [8]:

numeric_columns = ['Quantity', 'Price Per Unit', 'Total Spent']

for column in numeric_columns:
    clean_df[column] = pd.to_numeric(
        clean_df[column],
        errors='coerce'
    )

print(clean_df[numeric_columns].dtypes)


Quantity          float64
Price Per Unit    float64
Total Spent       float64
dtype: object


In [9]:

# Calculate expected total
clean_df['Calculated Total'] = (
    clean_df['Quantity'] * clean_df['Price Per Unit']
)

# Compare with existing Total Spent
valid_rows = clean_df.dropna(
    subset=['Quantity', 'Price Per Unit', 'Total Spent']
)

mismatches = valid_rows[
    valid_rows['Total Spent'] != valid_rows['Calculated Total']
]

print("Total valid records:", len(valid_rows))
print("Mismatched records:", len(mismatches))


Total valid records: 8544
Mismatched records: 0


In [10]:

display(clean_df[numeric_columns].describe())


,Quantity,Price Per Unit,Total Spent
count,9521.000000,9467.000000,9498.000000
mean,3.028463,2.949984,8.924352
std,1.419007,1.278450,6.009919
min,1.000000,1.000000,1.000000
25%,2.000000,2.000000,4.000000
50%,3.000000,3.000000,8.000000
75%,4.000000,4.000000,12.000000
max,5.000000,5.000000,25.000000


In [11]:

# Fill missing Total Spent using business logic

clean_df['Total Spent'] = clean_df['Total Spent'].fillna(
    clean_df['Quantity'] * clean_df['Price Per Unit']
)

print("Missing Total Spent:",
      clean_df['Total Spent'].isnull().sum())


Missing Total Spent: 40


In [12]:

# Check remaining missing numeric values

for column in ['Quantity', 'Price Per Unit', 'Total Spent']:
    print(column, ":", clean_df[column].isnull().sum())


Quantity : 479
Price Per Unit : 533
Total Spent : 40


In [13]:

display(clean_df[['Quantity', 'Price Per Unit', 'Total Spent']].head(10))


,Quantity,Price Per Unit,Total Spent
0,2.0,2.0,4.0
1,4.0,3.0,12.0
2,4.0,1.0,4.0
3,2.0,5.0,10.0
4,2.0,2.0,4.0
5,5.0,4.0,20.0
6,3.0,3.0,9.0
7,4.0,4.0,16.0
8,5.0,3.0,15.0
9,5.0,4.0,20.0


In [14]:

categorical_columns = ['Item', 'Payment Method', 'Location']

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(clean_df[column].value_counts(dropna=False))



--- Item ---
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
NaN          969
Name: count, dtype: int64

--- Payment Method ---
Payment Method
NaN               3178
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: int64

--- Location ---
Location
NaN         3961
Takeaway    3022
In-store    3017
Name: count, dtype: int64


In [15]:

for column in categorical_columns:
    clean_df[column] = clean_df[column].apply(
        lambda x: x.strip().title() if isinstance(x, str) else x
    )

print("Categorical formatting standardized!")


Categorical formatting standardized!


In [16]:

for column in categorical_columns:
    print(column, ":", clean_df[column].isnull().sum())


Item : 969
Payment Method : 3178
Location : 3961


In [17]:

# Convert transaction date into datetime format

clean_df['Transaction Date'] = pd.to_datetime(
    clean_df['Transaction Date'],
    errors='coerce'
)

print("Data type:",
      clean_df['Transaction Date'].dtype)

print("Invalid or missing dates:",
      clean_df['Transaction Date'].isnull().sum())


Data type: datetime64[us]
Invalid or missing dates: 460


In [18]:

print("Earliest Transaction:",
      clean_df['Transaction Date'].min())

print("Latest Transaction:",
      clean_df['Transaction Date'].max())


Earliest Transaction: 2023-01-01 00:00:00
Latest Transaction: 2023-12-31 00:00:00


In [19]:

clean_df['Year'] = clean_df['Transaction Date'].dt.year
clean_df['Month'] = clean_df['Transaction Date'].dt.month
clean_df['Day'] = clean_df['Transaction Date'].dt.day

display(clean_df[['Transaction Date', 'Year', 'Month', 'Day']].head(10))


,Transaction Date,Year,Month,Day
0,2023-09-08,2023.0,9.0,8.0
1,2023-05-16,2023.0,5.0,16.0
2,2023-07-19,2023.0,7.0,19.0
3,2023-04-27,2023.0,4.0,27.0
4,2023-06-11,2023.0,6.0,11.0
5,2023-03-31,2023.0,3.0,31.0
6,2023-10-06,2023.0,10.0,6.0
7,2023-10-28,2023.0,10.0,28.0
8,2023-07-28,2023.0,7.0,28.0
9,2023-12-31,2023.0,12.0,31.0


In [20]:

# Count duplicate rows

duplicate_count = clean_df.duplicated().sum()

print("Total Duplicate Rows:", duplicate_count)


Total Duplicate Rows: 0


In [21]:

# Remove duplicate rows

clean_df.drop_duplicates(inplace=True)

print("Duplicates removed successfully!")
print("Updated Dataset Shape:", clean_df.shape)


Duplicates removed successfully!
Updated Dataset Shape: (10000, 12)


In [22]:

print("Remaining Duplicates:", clean_df.duplicated().sum())


Remaining Duplicates: 0


In [23]:

numeric_columns = ['Quantity', 'Price Per Unit', 'Total Spent']

for column in numeric_columns:

    Q1 = clean_df[column].quantile(0.25)
    Q3 = clean_df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = clean_df[
        (clean_df[column] < lower_limit) |
        (clean_df[column] > upper_limit)
    ]

    print(f"\n{column}")
    print("Lower Limit:", lower_limit)
    print("Upper Limit:", upper_limit)
    print("Outliers:", len(outliers))



Quantity
Lower Limit: -1.0
Upper Limit: 7.0
Outliers: 0

Price Per Unit
Lower Limit: -1.0
Upper Limit: 7.0
Outliers: 0

Total Spent
Lower Limit: -8.0
Upper Limit: 24.0
Outliers: 268


In [24]:

# Final data quality report

print("FINAL DATA QUALITY REPORT")
print("-------------------------")

print("Dataset Shape:", clean_df.shape)

print("\nMissing Values:")
print(clean_df.isnull().sum())

print("\nDuplicate Rows:",
      clean_df.duplicated().sum())

print("\nData Types:")
print(clean_df.dtypes)

print("\nStatistical Summary:")
display(clean_df.describe())


FINAL DATA QUALITY REPORT
-------------------------
Dataset Shape: (10000, 12)

Missing Values:
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent           40
Payment Method      3178
Location            3961
Transaction Date     460
Calculated Total     994
Year                 460
Month                460
Day                  460
dtype: int64

Duplicate Rows: 0

Data Types:
Transaction ID                 str
Item                           str
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
Calculated Total           float64
Year                       float64
Month                      float64
Day                        float64
dtype: object

Statistical Summary:


,Quantity,Price Per Unit,Total Spent,Transaction Date,Calculated Total,Year,Month,Day
count,9521.000000,9467.000000,9960.000000,9540,9006.000000,9540.0,9540.000000,9540.000000
mean,3.028463,2.949984,8.930924,2023-07-01 23:00:31.698113,8.936098,2023.0,6.524004,15.738679
min,1.000000,1.000000,1.000000,2023-01-01 00:00:00,1.000000,2023.0,1.000000,1.000000
25%,2.000000,2.000000,4.000000,2023-04-01 00:00:00,4.000000,2023.0,4.000000,8.000000
50%,3.000000,3.000000,8.000000,2023-07-02 00:00:00,8.000000,2023.0,7.000000,16.000000
75%,4.000000,4.000000,12.000000,2023-10-02 00:00:00,12.000000,2023.0,10.000000,23.000000
max,5.000000,5.000000,25.000000,2023-12-31 00:00:00,25.000000,2023.0,12.000000,31.000000
std,1.419007,1.278450,6.004475,NaN,5.994375,0.0,3.449446,8.761003


In [25]:

# Save cleaned dataset

clean_df.to_csv('cleaned_cafe_sales.csv', index=False)

print("Cleaned dataset saved successfully!")


Cleaned dataset saved successfully!


In [26]:

import os

print("File Location:", os.path.abspath('cleaned_cafe_sales.csv'))


File Location: C:\Users\goniy\Downloads\OSIBSIP\cleaned_cafe_sales.csv
